# 04 — Gold Layer: Business Metrics

**Goal:** turn the raw simulated battles into business-ready metrics that answer our core questions:

1. Which type has the highest real win rate?
2. Does Speed matter more than type advantage?
3. Do legendary Pokémon really win more?
4. Is there "power creep" between generations?
5. Does the simulated type matchup matrix match the official type chart?

**Input:** `data/silver/pokemon_silver/`, `data/silver/simulated_battles_silver/`, `data/silver/combats_silver/`
**Output:** one Parquet table per metric in `data/gold/`, ready for Power BI.

In [1]:
import os
import sys

import findspark
findspark.init()

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType

spark = (
    SparkSession.builder
    .appName("PokemonGoldMetrics")
    .master("local[*]")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

SILVER_PATH = "../data/silver"
GOLD_PATH = "../data/gold"

battle_engine_path = os.path.abspath(os.path.join("..", "src", "battle_engine.py"))
spark.sparkContext.addPyFile(battle_engine_path)
sys.path.append(os.path.abspath(os.path.join("..", "src")))
from battle_engine import type_effectiveness

type_effectiveness_udf = F.udf(type_effectiveness, DoubleType())

print("Spark session ready.")

Spark session ready.


In [2]:
pokemon_dim = spark.read.parquet(f"{SILVER_PATH}/pokemon_silver").select(
    "pokemon_id", "name", "type_1", "type_2", "generation", "legendary",
    "hp", "attack", "defense", "sp_atk", "sp_def", "speed",
)

battles = spark.read.parquet(f"{SILVER_PATH}/simulated_battles_silver")

print(f"Pokemon: {pokemon_dim.count()}")
print(f"Simulated battles: {battles.count():,}")

Pokemon: 800
Simulated battles: 6,392,000


## Enrich battles with both participants' attributes

`simulated_battles_silver` only stores IDs — everything else comes from joining back to `pokemon_dim`, twice (once per side of the battle).

In [3]:
dim_a = pokemon_dim.select([F.col(c).alias(f"{c}_a") for c in pokemon_dim.columns])
dim_b = pokemon_dim.select([F.col(c).alias(f"{c}_b") for c in pokemon_dim.columns])

battles_enriched = (
    battles
    .join(dim_a, on="pokemon_id_a")
    .join(dim_b, on="pokemon_id_b")
    .withColumn("a_won", F.col("winner_id") == F.col("pokemon_id_a"))
)

battles_enriched.cache()
print(f"Enriched battles: {battles_enriched.count():,}")

Enriched battles: 6,392,000


## Long-format participants table

One row per Pokémon per battle (2 rows per battle), each tagged with whether that Pokémon won. This is what makes "win rate by type / generation / legendary" a simple `groupBy`.

In [4]:
participants_a = (
    battles_enriched
    .select(
        F.col("pokemon_id_a").alias("pokemon_id"),
        F.col("name_a").alias("name"),
        F.col("type_1_a").alias("type_1"),
        F.col("type_2_a").alias("type_2"),
        F.col("generation_a").alias("generation"),
        F.col("legendary_a").alias("legendary"),
        F.col("a_won").alias("won"),
    )
)

participants_b = (
    battles_enriched
    .select(
        F.col("pokemon_id_b").alias("pokemon_id"),
        F.col("name_b").alias("name"),
        F.col("type_1_b").alias("type_1"),
        F.col("type_2_b").alias("type_2"),
        F.col("generation_b").alias("generation"),
        F.col("legendary_b").alias("legendary"),
        (~F.col("a_won")).alias("won"),
    )
)

participants = participants_a.unionByName(participants_b)
participants.cache()
print(f"Participant rows: {participants.count():,}")

Participant rows: 12,784,000


## 1 — Win rate by type

Uses primary type only (`type_1`) — a Pokémon with a second type is counted once, under its primary type. Documented simplification, consistent with how we assign STAB in the damage formula.

In [5]:
win_rate_by_type = (
    participants
    .groupBy("type_1")
    .agg(
        F.count("*").alias("total_battles"),
        F.sum(F.col("won").cast("int")).alias("wins"),
    )
    .withColumn("win_rate_pct", F.round(F.col("wins") / F.col("total_battles") * 100, 2))
    .orderBy(F.desc("win_rate_pct"))
)

win_rate_by_type.show(20, truncate=False)
win_rate_by_type.write.mode("overwrite").parquet(f"{GOLD_PATH}/win_rate_by_type")

+--------+-------------+------+------------+
|type_1  |total_battles|wins  |win_rate_pct|
+--------+-------------+------+------------+
|Dragon  |511360       |359671|70.34       |
|Flying  |63920        |40948 |64.06       |
|Steel   |431460       |273165|63.31       |
|Dark    |495380       |277019|55.92       |
|Rock    |703120       |384879|54.74       |
|Fire    |830960       |449553|54.1        |
|Psychic |910860       |488863|53.67       |
|Ghost   |511360       |268372|52.48       |
|Electric|703120       |355882|50.61       |
|Water   |1789760      |900253|50.3        |
|Ground  |511360       |255098|49.89       |
|Fairy   |271660       |135167|49.76       |
|Ice     |383520       |188933|49.26       |
|Fighting|431460       |210045|48.68       |
|Grass   |1118600      |504999|45.15       |
|Normal  |1566040      |682051|43.55       |
|Poison  |447440       |189026|42.25       |
|Bug     |1102620      |428076|38.82       |
+--------+-------------+------+------------+



## 2 — Does Speed matter more than type advantage?

For each battle, classify Pokémon A's type advantage against B (immune / resisted / neutral / super effective), then cross it with whether A was faster. If speed dominates, we'd expect the faster Pokémon to keep winning even in the "resisted" bucket.

In [6]:
battles_enriched = battles_enriched.withColumn(
    "a_type_multiplier", type_effectiveness_udf(F.col("type_1_a"), F.col("type_1_b"), F.col("type_2_b"))
)

battles_enriched = battles_enriched.withColumn(
    "a_type_advantage",
    F.when(F.col("a_type_multiplier") == 0, "immune")
     .when(F.col("a_type_multiplier") < 1, "resisted")
     .when(F.col("a_type_multiplier") == 1, "neutral")
     .otherwise("super_effective"),
).withColumn("a_faster", F.col("speed_a") > F.col("speed_b"))

speed_vs_type_advantage = (
    battles_enriched
    .groupBy("a_type_advantage", "a_faster")
    .agg(
        F.count("*").alias("total_battles"),
        F.sum(F.col("a_won").cast("int")).alias("a_wins"),
    )
    .withColumn("a_win_rate_pct", F.round(F.col("a_wins") / F.col("total_battles") * 100, 2))
    .orderBy("a_type_advantage", F.desc("a_faster"))
)

speed_vs_type_advantage.show(20, truncate=False)
speed_vs_type_advantage.write.mode("overwrite").parquet(f"{GOLD_PATH}/speed_vs_type_advantage")

+----------------+--------+-------------+-------+--------------+
|a_type_advantage|a_faster|total_battles|a_wins |a_win_rate_pct|
+----------------+--------+-------------+-------+--------------+
|immune          |true    |127680       |24640  |19.3          |
|immune          |false   |100560       |12380  |12.31         |
|neutral         |true    |1725520      |1225387|71.02         |
|neutral         |false   |1851800      |533420 |28.81         |
|resisted        |true    |717640       |292581 |40.77         |
|resisted        |false   |826980       |105262 |12.73         |
|super_effective |true    |499500       |453681 |90.83         |
|super_effective |false   |542320       |295411 |54.47         |
+----------------+--------+-------------+-------+--------------+



## 3 — Legendary vs. non-legendary win rate

In [7]:
win_rate_by_legendary = (
    participants
    .groupBy("legendary")
    .agg(
        F.count("*").alias("total_battles"),
        F.sum(F.col("won").cast("int")).alias("wins"),
    )
    .withColumn("win_rate_pct", F.round(F.col("wins") / F.col("total_battles") * 100, 2))
    .orderBy(F.desc("legendary"))
)

win_rate_by_legendary.show(truncate=False)
win_rate_by_legendary.write.mode("overwrite").parquet(f"{GOLD_PATH}/win_rate_by_legendary")

+---------+-------------+-------+------------+
|legendary|total_battles|wins   |win_rate_pct|
+---------+-------------+-------+------------+
|true     |1038700      |847137 |81.56       |
|false    |11745300     |5544863|47.21       |
+---------+-------------+-------+------------+



## 4 — Power creep across generations

In [8]:
win_rate_by_generation = (
    participants
    .groupBy("generation")
    .agg(
        F.count("*").alias("total_battles"),
        F.sum(F.col("won").cast("int")).alias("wins"),
    )
    .withColumn("win_rate_pct", F.round(F.col("wins") / F.col("total_battles") * 100, 2))
    .orderBy("generation")
)

win_rate_by_generation.show(truncate=False)
win_rate_by_generation.write.mode("overwrite").parquet(f"{GOLD_PATH}/win_rate_by_generation")

+----------+-------------+-------+------------+
|generation|total_battles|wins   |win_rate_pct|
+----------+-------------+-------+------------+
|1         |2652680      |1277346|48.15       |
|2         |1693880      |793971 |46.87       |
|3         |2556800      |1231019|48.15       |
|4         |1933580      |1050835|54.35       |
|5         |2636700      |1363074|51.7        |
|6         |1310360      |675755 |51.57       |
+----------+-------------+-------+------------+



## 5 — Simulated type matchup matrix vs. the official type chart

For every `type_1_a` vs `type_1_b` combination, compute A's observed win rate from simulated battles, and compare it against the official effectiveness multiplier from `battle_engine.TYPE_CHART`. High agreement validates that the simulation is behaving the way the real game mechanics would predict.

In [9]:
type_matchup_matrix = (
    battles_enriched
    .groupBy("type_1_a", "type_1_b")
    .agg(
        F.count("*").alias("total_battles"),
        F.sum(F.col("a_won").cast("int")).alias("a_wins"),
    )
    .withColumn("a_win_rate_pct", F.round(F.col("a_wins") / F.col("total_battles") * 100, 2))
    .withColumn(
        "official_multiplier",
        type_effectiveness_udf(F.col("type_1_a"), F.col("type_1_b"), F.lit("None")),
    )
    .orderBy("type_1_a", "type_1_b")
)

type_matchup_matrix.show(20, truncate=False)
type_matchup_matrix.write.mode("overwrite").parquet(f"{GOLD_PATH}/type_matchup_matrix")

+--------+--------+-------------+------+--------------+-------------------+
|type_1_a|type_1_b|total_battles|a_wins|a_win_rate_pct|official_multiplier|
+--------+--------+-------------+------+--------------+-------------------+
|Bug     |Bug     |46920        |22125 |47.15         |1.0                |
|Bug     |Dark    |28960        |12729 |43.95         |2.0                |
|Bug     |Dragon  |30580        |6362  |20.8          |1.0                |
|Bug     |Electric|32760        |10168 |31.04         |1.0                |
|Bug     |Fairy   |15560        |3669  |23.58         |0.5                |
|Bug     |Fighting|20420        |8228  |40.29         |0.5                |
|Bug     |Fire    |34720        |2283  |6.58          |0.5                |
|Bug     |Flying  |5360         |767   |14.31         |0.5                |
|Bug     |Ghost   |31340        |8342  |26.62         |0.5                |
|Bug     |Grass   |49320        |36606 |74.22         |2.0                |
|Bug     |Gr

## Bonus — does simulation scale actually matter?

Compare win rate by type from the **original 50,000-battle dataset** (`combats_silver`) against our **millions-scale simulation**. With a small sample, rare matchups have noisy, unstable win rates; at scale, they stabilize. This is a genuinely useful data engineering point: more (simulated) data isn't just "more rows" — it changes the statistical confidence of the answer.

In [10]:
combats_original = spark.read.parquet(f"{SILVER_PATH}/combats_silver")

orig_a = (
    combats_original
    .join(pokemon_dim.select(F.col("pokemon_id").alias("first_pokemon_id"), F.col("type_1")), on="first_pokemon_id")
    .withColumn("won", F.col("first_pokemon_won"))
    .select("type_1", "won")
)
orig_b = (
    combats_original
    .join(pokemon_dim.select(F.col("pokemon_id").alias("second_pokemon_id"), F.col("type_1")), on="second_pokemon_id")
    .withColumn("won", ~F.col("first_pokemon_won"))
    .select("type_1", "won")
)

win_rate_by_type_original = (
    orig_a.unionByName(orig_b)
    .groupBy("type_1")
    .agg(F.count("*").alias("total_battles_original"), F.sum(F.col("won").cast("int")).alias("wins_original"))
    .withColumn("win_rate_pct_original", F.round(F.col("wins_original") / F.col("total_battles_original") * 100, 2))
)

comparison = (
    win_rate_by_type
    .join(win_rate_by_type_original, on="type_1", how="left")
    .select(
        "type_1",
        "total_battles_original", "win_rate_pct_original",
        F.col("total_battles").alias("total_battles_simulated"),
        F.col("win_rate_pct").alias("win_rate_pct_simulated"),
    )
    .orderBy(F.desc("win_rate_pct_simulated"))
)

comparison.show(20, truncate=False)
comparison.write.mode("overwrite").parquet(f"{GOLD_PATH}/win_rate_comparison_original_vs_simulated")

+--------+----------------------+---------------------+-----------------------+----------------------+
|type_1  |total_battles_original|win_rate_pct_original|total_battles_simulated|win_rate_pct_simulated|
+--------+----------------------+---------------------+-----------------------+----------------------+
|Dragon  |3932                  |63.33                |511360                 |70.34                 |
|Flying  |478                   |75.73                |63920                  |64.06                 |
|Steel   |3546                  |42.95                |431460                 |63.31                 |
|Dark    |3845                  |63.64                |495380                 |55.92                 |
|Rock    |5669                  |40.55                |703120                 |54.74                 |
|Fire    |6552                  |58.03                |830960                 |54.1                  |
|Psychic |7320                  |54.62                |910860            

In [11]:
spark.stop()